## Install Libraries

Runs the end-to-end stage of the GRF pipeline (architecture §8, §10) on the
candidate files produced by `GRF/ranking.ipynb`: load each saved candidate file
from Google Drive, apply the configured re-ranker (Top-1 or a language model),
and report end-to-end accuracy and `acc@k` per scenario, with one result file
per scenario.

This notebook does **not** load the encoder, the vector index, or the KB
artifact, and never re-runs retrieval. It needs no GPU unless
`LLM_BACKEND = "local"` and `LLM_LOCAL_DEVICE = "cuda"`.

Sources: `docs/architecture.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.

`tqdm` drives the progress bars; `transformers` (and Colab's `torch`) back the
local re-ranker and are unused when `RERANK = "top1"`. `pyyaml` reads the
re-ranker prompt from `source/GRF/prompts.yaml`.


In [ ]:
%pip install "tqdm>=4.66" "transformers>=4.44" "pyyaml>=6.0"


## Import Libraries

Third-party imports only. The `common` and `GRF` packages live inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

## Constants and Configuration

Candidates and results share one folder — `COMMON_DIR/output/{METHOD}/` with
`METHOD = "grf"` — because the input is the retrieval output:

- `CANDIDATES_DIR` + `CANDIDATES_GLOB`: every candidate JSONL written by
  `ranking.ipynb`, one per scenario,
- `RESULTS_DIR`: one end-to-end result JSON per scenario,
- `SUMMARY_PATH`: all scenarios in one table.

The KB CSV in `COMMON_DIR/data/` is read only for preferred names when
`RERANK = "llm"`.

`RERANK = "top1"` uses `PickTopOne`; `"llm"` uses a language model to pick one
CUI and falls back to Top-1 on any error (architecture §8). `ACC_K` is the
acc@k set computed on the re-ranked list; `k` must not exceed the candidate
count saved by `ranking.ipynb` (`top_k_concept`).

**Re-ranker model.** `LLM_BACKEND = "local"` loads `LLM_MODEL_LOCAL` through
`transformers` (Hub id or a local directory); `"api"` uses the
OpenAI-compatible client and needs `OPENAI_API_KEY`.

The re-ranker decodes `LLM_LOCAL_BATCH_SIZE` prompts at a time: every mention's
candidate list is turned into one prompt, and the whole chunk is decoded in a
single padded `generate` call, so the GPU is fed a batch instead of one short
sequence per mention. 16 fits an 80 GB card with the 32B weights loaded;
lower it if the card also holds another model or a longer candidate list runs
out of memory.


In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive layout: candidates come from ranking.ipynb; results go beside them
# ---------------------------------------------------------------------------
METHOD = "grf"
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
COMMON_DIR = DRIVE_ROOT / "work" / "nlp" / "UMLS" / "entity_linking"
DRIVE_DATA_DIR = COMMON_DIR / "data"
OUTPUT_DIR = COMMON_DIR / "output" / METHOD

CANDIDATES_DIR = OUTPUT_DIR / "candidates"                        # from ranking.ipynb
CANDIDATES_GLOB = "grf_*_candidates.jsonl"                        # every scenario
RESULTS_DIR = OUTPUT_DIR / "e2e_results"                          # one JSON per scenario
SUMMARY_PATH = OUTPUT_DIR / "grf_e2e_summary.json"                # all scenarios
KB_CSV_PATH = DRIVE_DATA_DIR / "kb.csv"                     # only for RERANK="llm"

# ---------------------------------------------------------------------------
# Re-ranking (architecture §8)
# ---------------------------------------------------------------------------
RERANK = "llm"                  # "top1" = paper's Pick Top-1 row; "llm" = real re-rank
ACC_K = (1, 5)                  # acc@k on the re-ranked list; k <= ranking's top_k_concept

# ---------------------------------------------------------------------------
# Re-ranker model: local HuggingFace weights (default) or the OpenAI-compatible API
# ---------------------------------------------------------------------------
LLM_BACKEND = "local"           # "local" (transformers) or "api"

# --- "local": loaded through transformers; HF id or a local directory path ---
LLM_MODEL_LOCAL = "Qwen/Qwen3-32B"
LLM_LOCAL_DEVICE = "auto"       # "auto", "cuda", "cpu" or "cuda:0"
LLM_LOCAL_DTYPE = "auto"        # "auto", "float16", "bfloat16" or "float32"
LLM_LOCAL_MAX_NEW_TOKENS = 64   # the answer is one CUI, so 64 tokens is plenty
LLM_LOCAL_BATCH_SIZE = 16       # prompts decoded together; fits an 80 GB card
LLM_LOCAL_TEMPERATURE = 0.0     # 0.0 = greedy (deterministic)
LLM_LOCAL_TRUST_REMOTE_CODE = False
LLM_LOCAL_USE_CHAT_TEMPLATE = True   # False = feed the raw prompt

# --- "api": OpenAI-compatible HTTP endpoint; needs OPENAI_API_KEY ---
LLM_MODEL_API = "gpt-4o"
LLM_API_BASE_URL = ""           # empty -> OPENAI_BASE_URL env var / api.openai.com
LLM_API_TIMEOUT_S = 30.0
# Set the key in this runtime (never commit it):
#   import getpass, os; os.environ["OPENAI_API_KEY"] = getpass.getpass()

LLM_ACTIVE_MODEL = LLM_MODEL_LOCAL if LLM_BACKEND == "local" else LLM_MODEL_API

# ---------------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------------
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value

log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Candidates   : {CANDIDATES_DIR}/{CANDIDATES_GLOB}")
print(f"KB data      : {KB_CSV_PATH} (used only when RERANK='llm')")
print(f"Results out  : {RESULTS_DIR} + {SUMMARY_PATH.name}")
print(f"Re-rank      : {RERANK}")
if RERANK == "llm":
    print(f"Re-rank LLM  : backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL}")
    if LLM_BACKEND == "local":
        print(f"LLM batching : {LLM_LOCAL_BATCH_SIZE} prompt(s) per decode")


## Stage 1 — Clone or Update the GRF Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/` and the GRF modules from
`source/GRF/`. Re-running this cell updates the existing clone instead of
downloading it again.

In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.candidates import load_candidates
from common.evaluator import accuracy_at_k
from common.kb import load_concepts_csv

from GRF.candidates import rerank_candidates
from GRF.hf_llm import TransformersChatClient
from GRF.llm import OpenAICompatibleClient
from GRF.rerank import LLMReranker, PickTopOne

log(f"Repository ready at {REPO_CLONE_DIR}; common and GRF imported from {SOURCE_DIR}.")


def build_llm_client(backend, model):
    """Local transformers model or the OpenAI-compatible API client."""
    if backend == "local":
        return TransformersChatClient(
            model,
            device=LLM_LOCAL_DEVICE,
            dtype=LLM_LOCAL_DTYPE,
            max_new_tokens=LLM_LOCAL_MAX_NEW_TOKENS,
            temperature=LLM_LOCAL_TEMPERATURE,
            trust_remote_code=LLM_LOCAL_TRUST_REMOTE_CODE,
            use_chat_template=LLM_LOCAL_USE_CHAT_TEMPLATE,
            batch_size=LLM_LOCAL_BATCH_SIZE,
        )
    if backend == "api":
        if not os.environ.get("OPENAI_API_KEY"):
            log(
                "WARNING: OPENAI_API_KEY is not set. API calls will fail and "
                "every mention will fall back to Top-1."
            )
        return OpenAICompatibleClient(
            model, base_url=LLM_API_BASE_URL or None, timeout_s=LLM_API_TIMEOUT_S
        )
    raise ValueError(f"LLM_BACKEND must be 'local' or 'api', got {backend!r}")


log("LLM factory ready; re-rank backend=%s." % LLM_BACKEND)

## Stage 2 — Mount Google Drive and Locate Inputs

The candidate files live on Drive. This stage mounts Drive, lists every file
matching `CANDIDATES_GLOB`, checks the results directory, and validates the
summary path. The KB CSV is required only for `RERANK = "llm"`.


In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

if not CANDIDATES_DIR.is_dir():
    raise FileNotFoundError(
        f"Candidates directory not found: {CANDIDATES_DIR}. "
        "Run source/notebooks/GRF/ranking.ipynb first."
    )

candidates_paths = sorted(CANDIDATES_DIR.glob(CANDIDATES_GLOB))
if not candidates_paths:
    raise FileNotFoundError(
        f"No {CANDIDATES_GLOB} under {CANDIDATES_DIR}. "
        "Run source/notebooks/GRF/ranking.ipynb first."
    )
for path in candidates_paths:
    log(
        f"Candidates: {path.name} "
        f"({path.stat().st_size / 1024 ** 2:.2f} MB)"
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary_path = Path(require_configured(SUMMARY_PATH, "SUMMARY_PATH"))
if summary_path.suffix.lower() != ".json":
    raise ValueError(f"SUMMARY_PATH must end in .json, got {summary_path.name!r}.")
log(f"{len(candidates_paths)} candidate file(s); results -> {RESULTS_DIR}")

if RERANK == "llm":
    kb_path = Path(require_configured(KB_CSV_PATH, "KB_CSV_PATH"))
    if not kb_path.is_file():
        raise FileNotFoundError(f"KB CSV not found: {kb_path}")
    log(f"KB data: {kb_path} ({kb_path.stat().st_size / 1024 ** 2:.2f} MB)")
else:
    kb_path = None
    log("RERANK='top1': KB data not needed.")


## Stage 3 — Check the Candidate Files

`ranking.ipynb` writes one JSONL file per scenario. This stage only confirms the
list is usable; each file is loaded inside the re-ranking loop so one scenario's
samples are never held alongside another's.


In [ ]:
slugs = [path.stem.removesuffix("_candidates") for path in candidates_paths]
if len(set(slugs)) != len(slugs):
    raise ValueError(f"duplicate scenario names in {CANDIDATES_DIR}: {slugs}")
log(f"Scenarios: {', '.join(slugs)}")


## Stage 4 — Load Concept Names (LLM Re-ranker Only)

The LLM re-ranker sees preferred names from the KB data. With
`RERANK = "top1"`, names are not used and this stage is skipped.

In [ ]:
if RERANK == "llm":
    if kb_path is None:
        raise ValueError("RERANK='llm' requires KB_CSV_PATH.")
    log(f"Loading KB concept names from {kb_path} ...")
    started = time.time()
    with progress_bar(None, "Parsing KB records", "record") as report:
        kb_concepts = load_concepts_csv(kb_path, progress_callback=report)
    if not kb_concepts:
        raise ValueError(f"{kb_path} produced no KB concepts.")
    concept_names = {concept.cui: concept.name for concept in kb_concepts}
    log(
        f"Loaded {len(concept_names):,} preferred names "
        f"in {time.time() - started:.1f}s."
    )
else:
    concept_names = {}
    log("RERANK='top1': concept names are not needed.")

## Stage 5 — Build the Re-ranker

`PickTopOne` keeps the highest-scoring candidate (CUI-ascending tie-break).
`LLMReranker` asks the configured model to pick one CUI; a failed or ambiguous
answer falls back to Top-1 (architecture §6, §8).

The re-ranker client follows `LLM_BACKEND`: a local `transformers` model
(`LLM_MODEL_LOCAL`) or the OpenAI-compatible API. The local client has no
`timeout_s` (a local generation cannot be interrupted safely), so the timeout
argument is forwarded only to the API client.

`LLMReranker` exposes `rerank_batch`, which `rerank_candidates` uses when it is
available: every mention's prompt goes to the client at once and is decoded in
batches of `LLM_LOCAL_BATCH_SIZE`. Each answer is still parsed on its own, so one
unusable answer degrades that mention to Top-1 without touching the others, and
a batch-level failure falls back to the per-mention path.

In [ ]:
if RERANK == "llm":
    log(
        f"Re-rank LLM: backend={LLM_BACKEND} model={LLM_ACTIVE_MODEL!r} "
        f"(weights load lazily on the first call)."
    )
    llm = build_llm_client(LLM_BACKEND, LLM_ACTIVE_MODEL)
    reranker = LLMReranker(
        llm, timeout_s=None if LLM_BACKEND == "local" else LLM_API_TIMEOUT_S
    )
else:
    llm = None
    reranker = PickTopOne()

log(f"Re-ranker ready: {type(reranker).__name__}.")

## Stage 6 — Run End-to-End Re-ranking

`rerank_candidates` applies the re-ranker to the saved candidate lists. With an
LLM re-ranker it collects every mention's prompt, decodes them in batches, and
only then assigns the answers back to their mentions; with `PickTopOne` there is
nothing to batch and it stays a plain loop.

The loop below walks every candidate file from stage 2: load that scenario's
samples and responses, re-rank them, compute end-to-end accuracy and `acc@k`,
and write
`RESULTS_DIR/<scenario>_e2e.json` before moving on, so only one scenario's
candidate list is live at a time. Each completed mention reports through
`progress_callback`, so the bar reflects real completed work — it advances once
a batch of prompts has been decoded and its answers parsed, not once per
prompt.


In [ ]:
results = []
for candidates_path in candidates_paths:
    slug = candidates_path.stem.removesuffix("_candidates")
    log(f"{slug}: loading {candidates_path.name} ...")
    with progress_bar(None, f"{slug}: loading", "record") as report:
        samples, responses = load_candidates(candidates_path, progress_callback=report)
    if not samples:
        raise ValueError(f"{candidates_path} contains no candidate records.")
    empty = sum(1 for response in responses if not response.candidates)
    log(
        f"{slug}: {len(samples):,} samples, {len(responses) - empty:,} with "
        f"candidates, {empty:,} without."
    )

    with progress_bar(len(samples), "reranking", "mention") as report:
        started = time.time()
        reranked = rerank_candidates(
            samples,
            responses,
            reranker=reranker,
            concept_names=concept_names,
            progress_callback=report,
        )
    log(f"{slug}: re-ranked {len(reranked):,} mentions in {time.time() - started:.1f}s.")

    failures = [
        response.debug["rerank"]["error"]
        for response in reranked
        if isinstance(response.debug, dict)
        and (response.debug.get("rerank") or {}).get("error")
    ]
    if failures:
        log(f"WARNING: {slug}: {len(failures):,} re-rank failure(s); first: {failures[0]}")

    # End-to-end accuracy: the re-ranked CUI equals the gold CUI. `acc@k` uses
    # the re-ranked list, where `rerank_candidates` puts the chosen candidate
    # first, so acc@1 is exactly this accuracy.
    hits = {}
    counts = {}
    for sample, response in zip(samples, reranked):
        hits[sample.label] = hits.get(sample.label, 0) + (response.cui == sample.gold_cui)
        counts[sample.label] = counts.get(sample.label, 0) + 1
    overall = sum(hits.values()) / len(samples)
    by_label = {label: hits[label] / counts[label] for label in counts}
    acc_at = {}
    for k in ACC_K:
        overall_k, _ = accuracy_at_k(samples, reranked, k)
        acc_at[str(k)] = overall_k

    results_path = RESULTS_DIR / f"{slug}_e2e.json"
    payload = {
        "candidates_file": str(candidates_path),
        "n_samples": len(samples),
        "rerank": RERANK,
        "llm_backend": LLM_BACKEND if RERANK == "llm" else None,
        "llm_model": LLM_ACTIVE_MODEL if RERANK == "llm" else None,
        "rerank_failures": len(failures),
        "accuracy": overall,
        "acc@k": acc_at,
        "by_label": {str(label): value for label, value in by_label.items()},
    }
    results_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    saved = json.loads(results_path.read_text(encoding="utf-8"))
    assert saved == payload, slug

    results.append(
        {
            "scenario": slug,
            "candidates_file": str(candidates_path),
            "results_file": str(results_path),
            "n": len(samples),
            "rerank_failures": len(failures),
            "accuracy": overall,
            "acc@k": acc_at,
            "by_label": payload["by_label"],
        }
    )
    log(f"{slug}: acc={100 * overall:.2f}% ({len(samples):,} samples); saved {results_path.name}.")
log(f"Re-ranked and saved {len(results)} scenario(s).")


## Stage 7 — Summary Table and Summary JSON

One row per scenario with `acc@k` (architecture §10), then the same numbers are
written to `SUMMARY_PATH` and read back to verify the file. The per-scenario
JSON files written in stage 6 keep the per-label breakdowns.

`rerank_candidates` puts the chosen CUI first in the re-ranked list, so `acc@1`
is the end-to-end accuracy and `acc@k` for larger `k` asks whether the gold CUI
is still anywhere in the first `k` of that list.


In [ ]:
ks = [int(k) for k in results[0]["acc@k"]] if results else []
print(
    f"{'scenario':<22}{'n':>7}"
    + "".join(f"{f'acc@{k} %':>10}" for k in ks)
    + f"{'fail':>7}"
)
for row in results:
    print(
        f"{row['scenario']:<22}{row['n']:>7}"
        + "".join(f"{100 * row['acc@k'][str(k)]:>10.2f}" for k in ks)
        + f"{row['rerank_failures']:>7}"
    )

payload = {
    "metric": "acc@k",
    "rerank": RERANK,
    "llm_backend": LLM_BACKEND if RERANK == "llm" else None,
    "llm_model": LLM_ACTIVE_MODEL if RERANK == "llm" else None,
    "n_files": len(results),
    "scenarios": results,
}
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
saved = json.loads(summary_path.read_text(encoding="utf-8"))
assert saved == payload
log(f"Saved and verified {summary_path} ({len(results)} scenario(s)).")
log("End-to-end evaluation complete.")


## Stage 8 — Release Memory

The results file is written and verified, so unload the re-ranker's local
weights (if any), drop the candidate lists and the KB names, then empty the
CUDA cache.

In [ ]:
if "llm" in globals() and llm is not None and hasattr(llm, "unload"):
    llm.unload()
    log("Local re-ranker weights unloaded.")

for name in (
    "candidates_paths",
    "samples",
    "responses",
    "reranked",
    "concept_names",
    "kb_concepts",
    "reranker",
    "llm",
    "failures",
    "results",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass

log("E2E memory released (samples, responses, re-ranked responses).")